# 🚀 Train an AI to Land a Rocket

Teach a neural network to land a lunar lander using reinforcement learning (PPO) — right here in your browser, no setup needed.

**How to run:** Colab menu → *Runtime* → **Run all**. Each step runs top to bottom: watch a random crash, train the AI, evaluate it, watch it land, then break it on purpose with a bad reward.

In [ ]:
# Install the libraries (quiet mode)
%pip install -q "gymnasium[box2d]" stable-baselines3 moviepy

In [ ]:
# Sanity check: make sure everything imported fine
import gymnasium
import stable_baselines3
print("gymnasium:", gymnasium.__version__)
print("stable-baselines3:", stable_baselines3.__version__)
print("All good — ready to fly. 🚀")

## Step 1 — Watch a random rocket crash

Before any training, the rocket just flails. Let's record one random attempt so we have a "before" clip.

In [ ]:
import gymnasium as gym
from IPython.display import Video

# Render to raw pixels so we can save the clip
env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, _ = env.reset()

frames = []
done = False
while not done:
    action = env.action_space.sample()  # totally random flailing
    obs, reward, terminated, truncated, _ = env.step(action)
    frames.append(env.render())
    done = terminated or truncated
env.close()

# Stitch frames into an mp4 (works with moviepy v1 and v2)
try:
    from moviepy.editor import ImageSequenceClip
except ImportError:
    from moviepy import ImageSequenceClip
ImageSequenceClip(frames, fps=30).write_videofile("random_crash.mp4", logger=None)

Video("random_crash.mp4", embed=True)

## Step 2 — Train the AI (takes ~20 minutes)

We hand the game to PPO (a reinforcement-learning algorithm). It plays thousands of attempts and slowly figures out landing. The `TimePenaltyWrapper` (from `reward_wrapper.py` in this repo) adds a tiny per-step cost so it learns to land *quickly* instead of dawdling. Go grab a coffee ☕

In [ ]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor, load_results
from stable_baselines3.common.results_plotter import ts2xy

from reward_wrapper import TimePenaltyWrapper  # our shaping, from this repo

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

env = gym.make("LunarLander-v3")
env = TimePenaltyWrapper(env, time_penalty=0.01)
env = Monitor(env, "logs")  # logs per-episode rewards

model = PPO("MlpPolicy", env, verbose=1, seed=42)
model.learn(total_timesteps=1_000_000)
model.save("models/ppo_lunar_lander")
env.close()

# Mean reward of the last 100 episodes — watch it climb
results = load_results("logs")
_, rewards = ts2xy(results, "timesteps")
print(f"\nep_rew_mean (last 100 episodes): {rewards[-100:].mean():.1f}")
print("Model saved to models/ppo_lunar_lander.zip")

## Step 3 — Did it learn? Evaluate

Run 10 fresh attempts with the trained brain. Lunar Lander counts an average of **200+** as solved.

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from reward_wrapper import TimePenaltyWrapper

env = gym.make("LunarLander-v3")
env = TimePenaltyWrapper(env, time_penalty=0.01)  # same wrapper as training!

model = PPO.load("models/ppo_lunar_lander", env=env)
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)

print(f"Mean reward: {mean_reward:.1f} ± {std_reward:.1f} (10 episodes)")
print("SOLVED! 🎉" if mean_reward >= 200 else "Not solved yet — try training longer.")
env.close()

## Step 4 — Watch it land

Record the trained pilot and play the clip right here.

In [ ]:
import glob
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import PPO
from IPython.display import Video

from reward_wrapper import TimePenaltyWrapper

env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = TimePenaltyWrapper(env, time_penalty=0.01)
env = RecordVideo(env, video_folder="videos",
                  episode_trigger=lambda episode_id: True,
                  name_prefix="lunar-lander")

model = PPO.load("models/ppo_lunar_lander")
for ep in range(3):
    obs, _ = env.reset()
    done, total = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        done = terminated or truncated
    print(f"Episode {ep + 1}: reward = {total:.1f}")
env.close()  # flushes the video files to disk

Video(sorted(glob.glob("videos/*.mp4"))[-1], embed=True)

## Step 5 — Break it, then fix it (the reward-shaping experiment)

Here's the real lesson of this project. Open `reward_wrapper.py` in the repo and look at the three wrappers:

- **`TimePenaltyWrapper`** (−0.01/step) — what you just trained with. Hovering costs a little, so landing fast wins.
- **`HoverBonusWrapper`** (commented out) — the *broken* idea: +1 every frame for staying airborne. The agent learns to **hover forever and never land** — it farms infinite reward instead of solving your actual goal. This is called **reward hacking**.
- **`DeltaShapingWrapper`** — the fix: reward only *progress* toward the pad (potential-based shaping), which mathematically cannot be farmed.

Try it yourself: uncomment `HoverBonusWrapper`, swap it into Step 2, retrain, and watch the rocket refuse to land. Then switch to `DeltaShapingWrapper` and watch it behave. Keep both clips — that's a portfolio-grade demo of reward shaping.

## 🔁 Restart from scratch

Want to start over? In Colab: menu → *Runtime* → **Restart and run all**. That wipes the trained model, videos, logs, and outputs, then re-runs every step from the beginning — training from zero again.

Or run the cell below for a manual reset (deletes the outputs without restarting the runtime):

In [ ]:
# Manual reset: delete trained model, videos, and logs
!rm -rf models videos logs random_crash.mp4
print("Reset done — Run all again to start from scratch.")